# TRUEWATCH: YOLO11-s fine-tune on Kaggle (Phase 2)

One detector, one head, mixed day and IR data, trained in two stages of one lineage.
Stage 1 (`yolo11s_day.yaml`) is the bulk of training on the full corpus. Stage 2 (`yolo11s_ir.yaml`) starts from
stage 1's best weights, repeats LWIR images twice per epoch and lowers the learning rate. It is not a separate IR model.
The reasoning is in `training/README.md` and at the top of `training/configs/yolo11s_day.yaml`.

**Before you run it**

1. Settings: Accelerator = GPU T4 x2 or P100. Internet = On (the code is cloned from GitHub and the AMP check downloads a tiny model).
2. Add data: the private dataset built by `datasets/scripts/09_build_yolo_ds.py` (the folder that holds `images/`, `labels/`, `data.yaml`).
   Copy `datasets/processed/index/split.jsonl` into that dataset as `index/split.jsonl` before uploading; it lets the
   evaluation tell KAIST day frames from KAIST night frames and group frames by video.
3. From the second session on, also add the previous session's output (this notebook's own output, or the results dataset it publishes).
   The run resumes from it.
4. Optional secrets (Add-ons, Secrets): `KAGGLE_USERNAME` and `KAGGLE_KEY` publish the results dataset; `HF_TOKEN` and `HF_MODEL_REPO`
   (for example `your-name/truewatch-yolo11s`) upload the exported ONNX to the Hugging Face Hub. Nothing is uploaded without them.

A session is capped at 12 h. Training stops itself after `MAX_HOURS`, writes `run_state.json`, and the next session continues from the last checkpoint.

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

GIT_URL = "https://github.com/0XSreekar/True-Watch-AI.git"
GIT_REF = "main"                  # branch or tag that holds training/
WORK = Path("/kaggle/working")
CODE = WORK / "truewatch"
RUNS = WORK / "runs"               # run directories; kept by Save Version and re-attached next session
OUT = WORK / "outputs"             # packaged as the results dataset at the end
DEVICE = "0"                       # "0,1" trains on both T4s with DDP: faster, but less exercised by the smoke test
MAX_HOURS = 9.0                    # per-session budget; the 12 h cap must also cover evaluation and export
RUN_SMOKE_TEST = True              # a few minutes on the GPU: proves paths, resume and export before the long run
RUN_STAGE_2 = True

_last_bar = 0.0


def sh(cmd, check=True):
    """Run a shell command from the repo, streaming output; progress-bar lines are throttled to one a minute."""
    global _last_bar
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, cwd=str(CODE) if CODE.exists() else None,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        is_bar = ("it/s]" in line) or ("s/it]" in line)
        if is_bar and time.time() - _last_bar < 60:
            continue
        if is_bar:
            _last_bar = time.time()
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")
    return code


def read_state(stage):
    path = RUNS / stage / "run_state.json"
    return json.loads(path.read_text()) if path.exists() else {}

In [ ]:
sh("nvidia-smi --query-gpu=name,memory.total --format=csv; python --version; df -h /kaggle/working | tail -1", check=False)

In [ ]:
if not CODE.exists():
    code_rc = sh(f"git clone --depth 1 --branch {GIT_REF} {GIT_URL} {CODE}", check=False)
    if code_rc != 0:
        # No internet: fall back to a dataset that carries the repo (needs training/ and datasets/config/).
        local = sorted(glob.glob("/kaggle/input/*/training/scripts/train.py"))
        if not local:
            raise RuntimeError("could not clone the repository and no attached dataset contains training/scripts/train.py")
        shutil.copytree(Path(local[0]).parents[2], CODE)
sh("git log -1 --oneline 2>/dev/null || echo '(code copied from an attached dataset)'", check=False)
sh("pip install -q -r training/requirements.txt")
sh("python -c \"import ultralytics, torch; print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())\"")

## 1. Smoke test (minutes, not hours)

Builds a tiny synthetic dataset and runs the real scripts end to end: train, kill and resume, evaluate, sweep, export with the
torch-versus-onnxruntime parity check, and the CPU benchmark. If anything is wrong with paths, versions or the GPU, it fails here.

In [ ]:
if RUN_SMOKE_TEST:
    sh(f"python training/scripts/smoke_test.py --device {DEVICE.split(',')[0]} --workdir {WORK / 'smoke'}")

## 2. Dataset preflight and the training plan

`--dry-run` locates the dataset, checks it (class histogram, label ids, duplicate filenames that would pair an image with the wrong
labels) and prints the resolved plan without training.

In [ ]:
sh(f"python training/scripts/train.py --stage day --run-dir {RUNS / 'day'} --device {DEVICE} --dry-run")

## 3. Restore the previous session

Copies the most advanced earlier run for each stage from an attached output dataset. Does nothing on the first session.

In [ ]:
def restore(stage):
    dest = RUNS / stage
    if (dest / "weights").exists():
        print(f"{stage}: run directory already present at {dest}")
        return
    logs = [Path(p) for p in glob.glob(f"/kaggle/input/*/runs/{stage}/train_log.csv")]
    if not logs:
        print(f"{stage}: no earlier session attached, starting fresh")
        return
    best = max(logs, key=lambda p: sum(1 for _ in p.open())).parent
    shutil.copytree(best, dest)
    print(f"{stage}: restored {best} -> {dest}")


RUNS.mkdir(parents=True, exist_ok=True)
for stage in ("day", "ir"):
    restore(stage)

## 4. Stage 1: day (the bulk of training, COCO weights to the mixed corpus)

`--auto-resume` continues from the newest valid checkpoint if there is one. `last.pt` is written every epoch and copied atomically to
`last_good.pt`, so a session killed mid-write cannot cost more than one epoch.

In [ ]:
sh(f"python training/scripts/train.py --stage day --run-dir {RUNS / 'day'} --device {DEVICE} --auto-resume --max-hours {MAX_HOURS}", check=False)
state1 = read_state("day")
STAGE1_DONE = bool(state1.get("complete"))
print(json.dumps(state1, indent=2))
if not STAGE1_DONE:
    print("Stage 1 is not finished. Save Version, attach this notebook's output to the next session, and run again: it resumes.")

## 5. Stage 2: IR emphasis (weights from stage 1, LWIR repeated, lower learning rate)

In [ ]:
STAGE2_DONE = False
if STAGE1_DONE and RUN_STAGE_2:
    sh(f"python training/scripts/train.py --stage ir --run-dir {RUNS / 'ir'} --device {DEVICE} --auto-resume --max-hours {MAX_HOURS}", check=False)
    state2 = read_state("ir")
    STAGE2_DONE = bool(state2.get("complete"))
    print(json.dumps(state2, indent=2))
    if not STAGE2_DONE:
        print("Stage 2 is not finished. Save Version and resume in the next session.")
else:
    print("Stage 2 skipped:", "stage 1 not finished" if not STAGE1_DONE else "RUN_STAGE_2 is False")

## 6. Evaluate on the validation split, day and IR separately

The test split is sealed until Phase 11 and is not touched here. Validation was also used to pick the checkpoint, so these numbers are optimistic.

In [ ]:
FINAL_STAGE = "ir" if STAGE2_DONE else ("day" if STAGE1_DONE else None)
WEIGHTS = RUNS / FINAL_STAGE / "weights" / "best.pt" if FINAL_STAGE else None
GPU = DEVICE.split(",")[0]
RESULTS = CODE / "training" / "results"
if WEIGHTS and WEIGHTS.exists():
    sh(f"python training/scripts/evaluate.py --weights {WEIGHTS} --tag final --device {GPU}")
    hard_set = CODE / "datasets" / "manifests" / "hard_set.txt"
    if hard_set.exists():
        baseline = RESULTS / "hardset_baseline.json"
        flag = "" if baseline.exists() else "--write-baseline"
        sh(f"python training/scripts/eval_hardset.py --weights {WEIGHTS} --tag final --device {GPU} {flag}", check=False)
    else:
        print("datasets/manifests/hard_set.txt is not in the repository clone: commit the manifest from the Mac, then re-run this cell.")
    sh(f"python training/scripts/sweep_conf.py --preds {RESULTS / 'cache' / 'preds_final.npz'} --tag final")
else:
    print("No finished stage to evaluate yet.")

## 7. Export to ONNX (opset 17, dynamic batch) and check torch against onnxruntime

The parity line must read PASS (max absolute difference below 1e-3). The upload to the Hugging Face Hub happens only when the
`HF_TOKEN` and `HF_MODEL_REPO` secrets exist.

In [ ]:
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name, "")


if WEIGHTS and WEIGHTS.exists():
    hf_token, hf_repo = secret("HF_TOKEN"), secret("HF_MODEL_REPO")
    push = f"--push-to-hub {hf_repo}" if (hf_token and hf_repo) else ""
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
    sh(f"python training/scripts/export_onnx.py --weights {WEIGHTS} --tag final {push}")
    if not push:
        print("HF_TOKEN / HF_MODEL_REPO not set: nothing was uploaded.")

## 8. Metrics report

Renders `training/results/METRICS.md`: measured against the slide-5 targets, with a verdict per target. Latency rows come from the
`benchmark_*.json` files committed in the repository, each labelled with the host it was measured on.

In [ ]:
if WEIGHTS and WEIGHTS.exists():
    benches = " ".join(f"--benchmark {p}" for p in sorted(glob.glob(str(RESULTS / "benchmark_*.json"))))
    hard = f"--hardset {RESULTS / 'hardset_final.json'}" if (RESULTS / "hardset_final.json").exists() else ""
    sh(
        f"python training/scripts/make_metrics.py --eval {RESULTS / 'eval_final.json'} {hard} "
        f"--sweep {RESULTS / 'sweep_final.json'} --export {RESULTS / 'export_final.json'} "
        f"--train-log {RUNS / FINAL_STAGE / 'train_log.csv'} {benches}"
    )
    print((RESULTS / "METRICS.md").read_text()[:6000])

## 9. Package and publish the results dataset

Weights, checkpoints and logs go to a private Kaggle Dataset. Only the small metrics files and the model URL are meant to go back to git.

In [ ]:
if OUT.exists():
    shutil.rmtree(OUT)
(OUT / "runs").mkdir(parents=True)
for stage in ("day", "ir"):
    src = RUNS / stage
    if src.exists():
        shutil.copytree(src, OUT / "runs" / stage, ignore=shutil.ignore_patterns("*.jpg", "*.png", "epoch*.pt", "*.cache"))
if RESULTS.exists():
    shutil.copytree(RESULTS, OUT / "results", ignore=shutil.ignore_patterns("cache"))

user, key = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY")
if user and key:
    meta = {
        "title": "TRUEWATCH YOLO11-s runs",
        "id": f"{user}/truewatch-yolo11s-runs",
        "licenses": [{"name": "other"}],
        "subtitle": "Checkpoints, logs and metrics for the TRUEWATCH detector fine-tune",
    }
    (OUT / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
    os.environ.update(KAGGLE_USERNAME=user, KAGGLE_KEY=key)
    stamp = time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime())
    rc = sh(f"kaggle datasets version -p {OUT} -m 'session {stamp}' --dir-mode zip", check=False)
    if rc != 0:
        sh(f"kaggle datasets create -p {OUT} --dir-mode zip")
else:
    print("KAGGLE_USERNAME / KAGGLE_KEY not set. To keep the results: Save Version, open the version's Output tab, choose New Dataset.")
print(sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*") if p.is_file())[:60])

## 10. What goes back to the repository

Commit only `training/results/*.json`, `*.csv`, `METRICS.md` and the Hugging Face model URL. Never a `.pt` or `.onnx`.
Download the small files from the results dataset, copy them into `training/results/` on your machine, and read `METRICS.md` before quoting any figure.